# Exploratory Data Analysis — French Motor TPL Claims

This notebook explores the **freMTPL2** dataset (French Motor Third-Party Liability),
covering ~678k motor insurance policies. We examine target distributions, feature
characteristics, and key business relationships before modelling.

## 1. Setup & Data Loading

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from claims.data import load_fremtpl2, build_claims_dataset

# Plotting defaults
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 100

print('Libraries loaded successfully.')

In [ ]:
# Load raw frequency and severity tables from OpenML
print('Fetching freMTPL2 from OpenML (may take ~30s on first run)...')
freq, sev = load_fremtpl2()
print(f'Frequency table : {freq.shape}')
print(f'Severity table  : {sev.shape}')

In [ ]:
# Join tables and engineer target columns
df = build_claims_dataset(freq, sev)
print(f'Claims dataset shape: {df.shape}')
df.head(3)

## 2. Dataset Overview

In [ ]:
print('=== Shape ===')
print(f'Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}')
print()
print('=== Data Types ===')
print(df.dtypes)

In [ ]:
# Descriptive statistics for numeric columns
df.describe(include='all').T

In [ ]:
# Missing values audit
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
print('Missing Values:')
print(missing_df[missing_df['missing_count'] > 0].to_string())
if missing_df['missing_count'].sum() == 0:
    print('  No missing values in any column (except AvgSeverity, which is NaN by design for zero-claim policies).')

## 3. Target Distribution

In [ ]:
# Claim frequency distribution — strongly zero-inflated
print('=== Claim Occurrence (HasClaim) ===')
has_claim_counts = df['HasClaim'].value_counts()
has_claim_pct = (has_claim_counts / len(df) * 100).round(2)
print(pd.concat([has_claim_counts.rename('count'), has_claim_pct.rename('pct%')], axis=1).to_string())
print()
print('=== ClaimNb Distribution (among claimants) ===')
print(df[df['ClaimNb'] > 0]['ClaimNb'].value_counts().sort_index())

In [ ]:
# Exposure analysis
print('=== Exposure Summary ===')
print(df['Exposure'].describe().round(4))
print()
print(f'Total exposure-years : {df["Exposure"].sum():,.1f}')
print(f'Total claims         : {df["ClaimNb"].sum():,}')
print(f'Portfolio claim rate : {df["HasClaim"].mean():.4f} ({df["HasClaim"].mean()*100:.2f}%)')

In [ ]:
# Bar chart of HasClaim
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: count bar
labels = ['No Claim (0)', 'Has Claim (1)']
counts = [has_claim_counts.get(0, 0), has_claim_counts.get(1, 0)]
colors = ['#4C72B0', '#DD8452']
axes[0].bar(labels, counts, color=colors, edgecolor='white', linewidth=0.8)
for i, (lbl, cnt) in enumerate(zip(labels, counts)):
    axes[0].text(i, cnt + 2000, f'{cnt:,}\n({cnt/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=10)
axes[0].set_title('Claim Occurrence — HasClaim (Count)')
axes[0].set_ylabel('Number of Policies')
axes[0].set_ylim(0, max(counts) * 1.12)

# Right: ClaimNb distribution (zero-inflated)
claim_dist = df['ClaimNb'].value_counts().sort_index().head(6)
axes[1].bar(claim_dist.index.astype(str), claim_dist.values, color='#4C72B0', edgecolor='white')
axes[1].set_title('Zero-Inflated ClaimNb Distribution')
axes[1].set_xlabel('Number of Claims')
axes[1].set_ylabel('Policy Count')
for i, v in enumerate(claim_dist.values):
    axes[1].text(i, v + 1000, f'{v:,}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 4. Claim Amount Distribution

In [ ]:
# AvgSeverity: log-scale histogram (claimants only)
severity = df['AvgSeverity'].dropna()
print(f'Claimant policies (AvgSeverity not NaN): {len(severity):,}')
print(severity.describe().round(2))

In [ ]:
# Log-scale histogram of AvgSeverity
fig, ax = plt.subplots(figsize=(9, 4))
log_severity = np.log1p(severity)
ax.hist(log_severity, bins=60, color='#4C72B0', edgecolor='white', alpha=0.85)
ax.set_xlabel('log(1 + AvgSeverity) — Claim Amount')
ax.set_ylabel('Policy Count')
ax.set_title('AvgSeverity Distribution (Log Scale) — Claimant Policies Only')
# Add x-tick labels in original scale
tick_vals = [100, 500, 1000, 5000, 10000, 50000, 100000]
ax.set_xticks(np.log1p(tick_vals))
ax.set_xticklabels([f'{v:,}' for v in tick_vals], rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Box plot of AvgSeverity by VehBrand (top brands by volume)
top_brands = df['VehBrand'].value_counts().head(8).index.tolist()
plot_data = df[df['VehBrand'].isin(top_brands) & df['AvgSeverity'].notna()].copy()
plot_data['log_severity'] = np.log1p(plot_data['AvgSeverity'])

fig, ax = plt.subplots(figsize=(12, 5))
brand_order = (
    plot_data.groupby('VehBrand')['log_severity']
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)
sns.boxplot(
    data=plot_data, x='VehBrand', y='log_severity',
    order=brand_order, palette='muted', ax=ax,
    flierprops=dict(marker='o', markersize=2, alpha=0.3)
)
ax.set_xlabel('Vehicle Brand')
ax.set_ylabel('log(1 + AvgSeverity)')
ax.set_title('Claim Severity by Vehicle Brand (Top 8 by Volume)')
plt.tight_layout()
plt.show()

## 5. Feature Analysis

In [ ]:
# DrivAge distribution
fig, ax = plt.subplots(figsize=(9, 4))
driv_age = pd.to_numeric(df['DrivAge'], errors='coerce').dropna()
ax.hist(driv_age, bins=50, color='#4C72B0', edgecolor='white', alpha=0.85)
ax.axvline(driv_age.mean(), color='#DD8452', linestyle='--', lw=2, label=f'Mean: {driv_age.mean():.1f}')
ax.axvline(driv_age.median(), color='#55A868', linestyle=':', lw=2, label=f'Median: {driv_age.median():.1f}')
ax.set_xlabel('Driver Age (years)')
ax.set_ylabel('Policy Count')
ax.set_title('Driver Age Distribution')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# BonusMalus distribution
fig, ax = plt.subplots(figsize=(9, 4))
bm = pd.to_numeric(df['BonusMalus'], errors='coerce').dropna()
ax.hist(bm, bins=60, color='#DD8452', edgecolor='white', alpha=0.85)
ax.axvline(100, color='black', linestyle='--', lw=1.5, label='Base score = 100')
ax.set_xlabel('Bonus-Malus Score')
ax.set_ylabel('Policy Count')
ax.set_title('BonusMalus Distribution (100 = neutral; >100 = penalised)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Density map by Area (proxy for urban vs rural)
density_by_area = (
    df.groupby('Area')['Density']
    .agg(median_density='median', policy_count='count')
    .sort_values('median_density', ascending=False)
    .reset_index()
)
print('Median Density & Policy Count by Area:')
print(density_by_area.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(
    density_by_area['Area'],
    density_by_area['median_density'],
    color=sns.color_palette('Blues_d', len(density_by_area)),
    edgecolor='white'
)
ax.set_xlabel('Area Code')
ax.set_ylabel('Median Population Density')
ax.set_title('Population Density by Area Code (A=most urban, F=most rural)')
ax.set_yscale('log')
plt.tight_layout()
plt.show()

## 6. Bivariate Analysis

In [ ]:
# Claim rate by VehBrand
claim_rate_brand = (
    df.groupby('VehBrand')['HasClaim']
    .agg(claim_rate='mean', n_policies='count')
    .sort_values('claim_rate', ascending=False)
    .reset_index()
)
print('Claim Rate by Vehicle Brand:')
print(claim_rate_brand.to_string(index=False))

In [ ]:
# Claim rate by Area
claim_rate_area = (
    df.groupby('Area')['HasClaim']
    .agg(claim_rate='mean', n_policies='count')
    .sort_values('claim_rate', ascending=False)
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: claim rate by VehBrand
top_brands_cr = claim_rate_brand.head(10)
axes[0].barh(top_brands_cr['VehBrand'], top_brands_cr['claim_rate'], color='#4C72B0')
axes[0].axvline(df['HasClaim'].mean(), color='red', linestyle='--', lw=1.5, label=f'Portfolio avg: {df["HasClaim"].mean():.3f}')
axes[0].set_xlabel('Claim Rate')
axes[0].set_title('Claim Rate by Vehicle Brand (Top 10)')
axes[0].legend(fontsize=9)

# Right: claim rate by Area
axes[1].bar(claim_rate_area['Area'], claim_rate_area['claim_rate'], color='#DD8452')
axes[1].axhline(df['HasClaim'].mean(), color='red', linestyle='--', lw=1.5, label=f'Portfolio avg: {df["HasClaim"].mean():.3f}')
axes[1].set_xlabel('Area Code')
axes[1].set_ylabel('Claim Rate')
axes[1].set_title('Claim Rate by Area')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Claim rate by DrivAge bucket
df['DrivAge_num'] = pd.to_numeric(df['DrivAge'], errors='coerce')
df['AgeBucket'] = pd.cut(
    df['DrivAge_num'],
    bins=[17, 25, 35, 45, 55, 65, 100],
    labels=['18-25', '26-35', '36-45', '46-55', '56-65', '65+']
)
claim_rate_age = (
    df.groupby('AgeBucket', observed=True)['HasClaim']
    .agg(claim_rate='mean', n_policies='count')
    .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(claim_rate_age['AgeBucket'].astype(str), claim_rate_age['claim_rate'], color='#55A868', edgecolor='white')
ax.axhline(df['HasClaim'].mean(), color='red', linestyle='--', lw=1.5, label=f'Portfolio avg: {df["HasClaim"].mean():.3f}')
ax.set_xlabel('Driver Age Bucket')
ax.set_ylabel('Claim Rate')
ax.set_title('Claim Rate by Driver Age Group')
ax.legend()
for i, row in claim_rate_age.iterrows():
    ax.text(i, row['claim_rate'] + 0.001, f"{row['claim_rate']:.3f}", ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap of numeric features vs HasClaim
numeric_cols = ['VehAge', 'DrivAge_num', 'BonusMalus', 'Density', 'Exposure', 'HasClaim', 'ClaimFrequency']
corr_data = df[numeric_cols].copy()
for col in ['BonusMalus', 'Density']:
    corr_data[col] = pd.to_numeric(corr_data[col], errors='coerce')
corr_matrix = corr_data.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, ax=ax,
    annot_kws={'size': 9}
)
ax.set_title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## 7. Key Business Insights

Based on the exploratory analysis above, five key findings stand out:

1. **Extreme class imbalance — zero-inflated portfolio.** Approximately 93-94% of policies generate zero claims in a given year. Any model must account for this through class weighting, oversampling, or specialised loss functions. A naive accuracy baseline of ~93% offers no real predictive signal.

2. **BonusMalus is the strongest individual predictor.** The bonus-malus score encodes years of accumulated claims history. Policies with BonusMalus > 100 (penalised drivers) show claim rates materially above the portfolio average (~7%), confirming it as a primary discriminating feature. This is consistent with actuarial pricing theory where experience-rated premiums capture individual risk.

3. **Young drivers (18-25) carry the highest claim rate.** The 18-25 age group exhibits the highest observed claim frequency — a well-known actuarial phenomenon driven by inexperience. The 65+ cohort also shows elevated rates relative to the 36-65 middle band, consistent with reduced reaction times at older ages.

4. **Urban areas (code A/B) show higher claim frequency than rural areas.** Higher population density correlates with more traffic exposure, greater collision risk, and higher claim rates. Area code is therefore a meaningful geographic risk factor, complementing the raw Density variable.

5. **Severity is highly skewed with a long right tail.** AvgSeverity on a log scale appears roughly log-normal, but extreme outlier claims (>€50k) are present. This means mean severity is a poor central estimate; median or log-normal GLM severity models will be necessary for pricing and reserving.